# LoRA fine-tune of Llama 3.2 1B Instruct (WP8)

Trains a LoRA adapter on the knowledge-base-written answers. Only training runs here; merging,
conversion, export and evaluation run on the laptop (see `models/llm/lora/README.md`).

**Before you start**
1. On Hugging Face, open `meta-llama/Llama-3.2-1B-Instruct` and request access (accepting the
   Llama 3.2 Community License). Wait for the approval email.
2. Create a Hugging Face access token with read access.
3. On the laptop: `uv run python models/llm/lora/make_bundle.py`, which writes
   `models/llm/lora/out/lora_bundle.zip`.
4. Runtime → Change runtime type → a GPU (T4 is enough).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install (versions pinned to match the laptop's `finetune` group)

In [ ]:
!pip install -q uv
!uv pip install --system -q "transformers==5.22.2" "peft==0.21.2" \
    "accelerate==1.15.0" "pytorch-tokenizers==1.6.0"

## 2. Upload `lora_bundle.zip`

In [ ]:
import zipfile

from google.colab import files

uploaded = files.upload()
zipfile.ZipFile(next(iter(uploaded))).extractall("work")
!ls -la work work/data && wc -l work/data/*.jsonl

## 3. Download the base model (needs the approved access and your token)

In [ ]:
from huggingface_hub import login, snapshot_download

login()  # paste the token when asked
base = snapshot_download(
    "meta-llama/Llama-3.2-1B-Instruct",
    local_dir="base",
    allow_patterns=["*.json", "*.safetensors", "original/tokenizer.model"],
)
!ls -la base base/original

## 4. Train

About 470 examples, 2 epochs. Training uses the same C++ tokenizer as the ExecuTorch runner on
the phone (`pytorch_tokenizers.CppTiktoken`), and the loss covers the answer tokens only.

In [ ]:
!cd work && python train.py --base ../base --tokenizer ../base/original/tokenizer.model \
    --data data --out ../run1 --epochs 2 --batch 8 --lr 2e-4 --grad-checkpointing

In [ ]:
import json

log = json.load(open("run1/train_log.json"))
print("val loss:", [round(v["loss"], 3) for v in log["val_loss"]])
print("last train losses:", [round(t["loss"], 3) for t in log["train_loss"][-5:]])

## 5. Download the adapter

The adapter is small (tens of MB). On the laptop, continue with `models/llm/lora/README.md`:
merge + convert (`convert.py --check`), export (`export.py --mode int4`), then score
(`eval/llm_quality/run.py`).

In [ ]:
!cd run1 && zip -qr ../run1_adapter.zip adapter train_log.json
files.download("run1_adapter.zip")